In [1]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "GT"

In [3]:
if base == "GT":
    sec = "Dino"
if base == "Dino":
    sec = "GT"

df_comp = pd.read_csv(f"GT Base Output/Comparison.csv")

M_comp = df_comp.to_numpy()
display(df_comp)

,ID (GT),ID (Dino),x (GT),y (GT),z (GT),x (Dino),y (Dino),z (Dino),t,t_start (GT),...,FI (GT),FI (Dino),Track Length (GT),Track Length (Dino),Feature 0,Feature 1,Feature 2,Distance,Multi ID (Dino),Multi Distance (Dino)
0,0,496,494.182,456.818,4.909,494.40220,457.05410,3.812911,0.0,0.0,...,1.0,3.002,32.0,8.0,7.050,-0.9290,0.6533,1.142647,496,1.14
1,0,496,488.909,461.545,5.273,488.72308,461.87270,3.851263,1.0,0.0,...,1.0,3.117,32.0,8.0,8.610,-0.4287,0.5693,1.470813,496,1.47
2,0,496,483.636,466.182,5.455,483.79166,465.98383,4.156463,2.0,0.0,...,1.0,3.014,32.0,8.0,8.460,-0.6533,0.0950,1.322762,496,1.32
3,0,496,478.909,471.455,5.545,478.95007,471.72382,4.149757,3.0,0.0,...,1.0,3.008,32.0,8.0,8.750,-1.6650,0.3303,1.421497,496,1.42
4,0,496,475.909,476.636,5.727,475.89853,476.68784,3.888299,4.0,0.0,...,1.0,3.613,32.0,8.0,10.445,0.5347,0.3210,1.839461,496,1.84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47558,2883,10501,63.727,184.909,2.091,64.15241,184.70474,2.313800,99.0,96.0,...,1.0,3.516,4.0,5.0,8.305,-0.7680,-0.5015,0.521858,10501,0.52
47559,2884,8235,239.818,506.455,3.000,240.08049,506.40567,2.599924,96.0,96.0,...,1.0,3.477,4.0,27.0,9.914,0.3694,-0.4734,0.481036,8235,0.48
47560,2884,10010,239.182,505.273,2.818,239.24835,505.07892,2.031062,97.0,96.0,...,1.0,3.795,4.0,11.0,8.670,1.1980,0.3325,0.813229,10010,0.81
47561,2884,10198,240.273,505.182,2.455,240.40868,495.68110,7.999989,98.0,96.0,...,1.0,-1.069,4.0,6.0,-1.615,-1.2910,-0.1690,11.001473,NaN,NaN


In [4]:
ID_vec = defaultdict(list)

for ID in M_comp:
    ID_vec[ID[0]].append(ID)

for ID_val in ID_vec:
    ID_vec[ID_val] = np.array(ID_vec[ID_val]) 

ID_list = np.zeros((len(ID_vec),5), dtype=object)

In [5]:
for i, ID in enumerate(ID_vec):
    ID_val = ID_vec[ID]
    track_coverage = np.sum(ID_val[:,18] < 3.5)
    missing_t = ID_val[ID_val[:, 18] > 3.5][:,8]
    ID_list[i][0] = ID_val[0][0]
    ID_list[i][1] = ID_val[0][13]
    ID_list[i][2] = track_coverage
    ID_list[i][3] = missing_t.tolist()
    block_starts = np.where(np.diff(ID_val[:,1]) != 0)[0] + 1
    block_starts = np.insert(block_starts, 0, 0)
    track_index = np.where(np.diff(ID_val[:,1]) != 0)[0] + 1
    track_index = np.insert(track_index, 0, 0)
    y_ID = ID_val[:,1][block_starts]
    track_lengths = ID_val[:,14][track_index]
    y = np.stack((y_ID,block_starts+1,track_lengths), axis=1)
    y_all = ';'.join([','.join(map(str, row)) for row in y])
    ID_list[i][4] = y_all

In [6]:
df_ID = pd.DataFrame(ID_list, columns = [f"ID ({base})",f"Track Length ({base})",f"Track Coverage","Missing Time Points",f"{sec} Data"])

In [7]:
df_ID.to_csv(f"GT Base Output/ID_Coverage.csv", index=False)